# 🚀 SAIR PyTorch Mastery - Lecture 5B: Segmentation, Pose & Gesture Control
## **From Boxes to Pixels to Interaction** 🇸🇩

**Course:** Ultimate Applied Deep Learning with PyTorch  
**Module:** Advanced Computer Vision - Segmentation, Pose, Gesture  
**Instructor:** Mohammed Awad Ahmed (Silva)  
**SAIR Community:** Building Sudan's AI Future

---

## 📋 PRE-LECTURE KNOWLEDGE CHECK

Before diving in, ensure you've completed Lecture 5A:

| Concept | Your Confidence (1-5) | Need Review? |
|---------|----------------------|--------------|
| YOLO detection | □ | □ |
| Bounding box formats | □ | □ |
| IoU calculation | □ | □ |
| NMS (Non-Maximum Suppression) | □ | □ |
| YOLO parameter tuning | □ | □ |

**If you scored <3 on any concept, review Lecture 5A first!**

## 📘 How to Use This Notebook

**Learning Outcomes:** After completing this notebook, you will be able to:
1. Segment objects with pixel-perfect masks
2. Remove backgrounds like Zoom/Teams
3. Track 17 body keypoints in real-time
4. Build gesture recognition systems
5. Control computers with hand movements
6. Collect high-quality training data with your phone
7. Train YOLO on YOUR custom data - THE #1 SKILL!
8. Build a complete gesture-controlled application

**Estimated Time:** 2-3 hours  
**Prerequisites:** Lecture 5A completed

## 📈 YOUR LEARNING JOURNEY TRACKER

| Part | Section | Status | Time | Confidence |
|------|---------|--------|------|------------|
| **Part 0** | Recap: What We Built in 5A | □ | __ min | __ |
| **Part 6** | Segmentation: From Boxes to Pixels | □ | __ min | __ |
| **🎥 DEMO 2** | Live Segmentation - "Remove My Background!" | □ | __ min | __ |
| **Part 6.5** | Pose Estimation: 17 Keypoints That See You | □ | __ min | __ |
| **🎥 DEMO 3** | Live Pose - "YOLO Sees My Joints!" | □ | __ min | __ |
| **Part 6.6** | Gesture Control: From Seeing to Understanding | □ | __ min | __ |
| **🎥 DEMO 4** | GRAND FINALE - "I Control with My Hand!" | □ | __ min | __ |
| **Part 7** | THE MAIN EVENT: Train YOLO on YOUR Data | □ | __ min | __ |
| **Part 7.5** | Data Collection Masterclass | □ | __ min | __ |
| **🎥 DEMO 5** | Victory Lap - "YOUR Model Detects Me!" | □ | __ min | __ |
| **Part 10** | Portfolio Projects & Final Challenge | □ | __ min | __ |
| **Final** | Mastery Checklist & Assessment | □ | __ min | __ |

**Goal:** Complete all parts with confidence ≥4/5

In [ ]:
# Initial Setup and Imports
import torch
import torchvision
import numpy as np
import cv2
import time
import warnings
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import os
import gc

warnings.filterwarnings('ignore')
print("✅ Basic imports loaded")

# Install/Import YOLO
try:
    from ultralytics import YOLO
    print("✅ YOLO already installed")
except:
    print("📦 Installing YOLO...")
    !pip install ultralytics
    from ultralytics import YOLO
    print("✅ YOLO installed")

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Download sample image
!wget -q -O street.jpg https://raw.githubusercontent.com/ultralytics/yolov5/master/data/images/bus.jpg
print("✅ Sample image downloaded: street.jpg")

# 🎯 PART 0: Recap - What We Built in 5A

## 📊 Your Journey So Far

```
┌─────────────────────────────────────────────────────────────────┐
│                   LECTURE 5A: COMPLETED ✅                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ✅ Detection (What + Where)                                   │
│     • Bounding boxes around objects                            │
│     • IoU, NMS, confidence thresholds                          │
│     • YOLO family (nano to xlarge)                             │
│                                                                 │
│  NOW: LECTURE 5B - GOING DEEPER                                │
│  ┌──────────────────────────────────────────────────────────┐  │
│  │ 🎨 SEGMENTATION → Pixel-perfect masks                    │  │
│  │ 🦾 POSE → 17 body keypoints                              │  │
│  │ ✨ GESTURE → Understanding movement                      │  │
│  │ 🎓 CUSTOM TRAINING → YOUR data, YOUR model              │  │
│  └──────────────────────────────────────────────────────────┘  │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Quick refresher - load detection model
detection_model = YOLO('yolov8n.pt')
print("✅ Detection model loaded (from 5A)")

# Quick detection test
results = detection_model('street.jpg')[0]
print(f"✅ Detected {len(results.boxes)} objects")

plt.figure(figsize=(10, 6))
plt.imshow(results.plot())
plt.axis('off')
plt.title("5A Recap: Detection with Bounding Boxes")
plt.show()

print("\n🎯 In 5A, we learned WHERE objects are.")
print("   In 5B, we'll learn WHICH PIXELS belong to them!")

## 🤔 The Natural Question

Now that we can draw boxes around objects, what's next?

**Think about it:**
- A box tells us "something is here"
- But WHAT EXACTLY is that something? A person? Their shadow? Both?

**Segmentation answers: "THESE EXACT pixels belong to the object"**

Let's see the difference...

# 🎨 PART 6: Segmentation - From Boxes to Pixels

## 6.1 What is Segmentation?

**Detection:** "There's a person at [100, 50, 300, 400]"  
**Segmentation:** "THESE EXACT PIXELS are the person"

In [ ]:
print("="*60)
print("PART 6: SEGMENTATION")
print("="*60)

# Load segmentation model
seg_model = YOLO('yolov8n-seg.pt')
print("✅ YOLOv8n-seg loaded")

# Run on image
seg_results = seg_model('street.jpg')[0]

# Visualize
plt.figure(figsize=(12, 8))
plt.imshow(seg_results.plot())
plt.axis('off')
plt.title(f'YOLO Segmentation: {len(seg_results.masks)} objects with pixel-perfect masks')
plt.show()

print("🎯 Each object now has a pixel-perfect mask!")
print("   This is how Zoom removes your background!")

## 🔍 Spot the Difference

Look closely at the image above. Notice how the masks follow the EXACT shape of each object:
- People: Masks follow arms, legs, head shape
- Cars: Masks follow curves, windows, wheels

**This precision is what makes background removal possible!**

## 6.2 Comparing Detection vs Segmentation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Detection
det_results = detection_model('street.jpg')[0]
axes[0].imshow(det_results.plot())
axes[0].set_title("Detection: Bounding Boxes", fontsize=14, fontweight='bold')
axes[0].axis('off')

# Segmentation
seg_results = seg_model('street.jpg')[0]
axes[1].imshow(seg_results.plot())
axes[1].set_title("Segmentation: Pixel Masks", fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.suptitle("Same Scene, Different Understanding", fontsize=16)
plt.tight_layout()
plt.show()

print("\n🔍 Notice the difference:")
print("   • Detection: Rectangular boxes (fast, simple)")
print("   • Segmentation: Exact object shape (slower, precise)")

## 💡 When to Use Each

| Task | Choose | Why |
|------|--------|-----|
| Security camera | Detection | Just need to know "person present" |
| Background removal | Segmentation | Need exact person shape |
| Self-driving car | Both! | Boxes for cars, masks for road |
| Medical imaging | Segmentation | Need exact tumor boundaries |

**Now let's build something useful with segmentation...**

## ✨ 6.3 Background Removal (Like Zoom/Teams)

In [ ]:
def remove_background(image_path, model, keep_class=None, bg_color=(255, 255, 255)):
    """Remove background, keep specified objects"""
    # Run segmentation
    results = model(image_path)[0]
    
    if len(results.masks) == 0:
        print("⚠️ No objects detected")
        return cv2.imread(image_path)
    
    # Load image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Combine masks
    combined_mask = np.zeros((h, w), dtype=bool)
    
    for i, (mask, cls) in enumerate(zip(results.masks.data, results.boxes.cls)):
        if keep_class is None or cls == keep_class:
            mask_np = mask.cpu().numpy()
            mask_resized = cv2.resize(mask_np, (w, h))
            combined_mask = np.logical_or(combined_mask, mask_resized > 0.5)
    
    # Apply mask
    result = img.copy()
    result[~combined_mask] = bg_color
    
    return result

# Test on image
result = remove_background('street.jpg', seg_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(cv2.cvtColor(cv2.imread('street.jpg'), cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(result)
axes[1].set_title('Background Removed')
axes[1].axis('off')

plt.suptitle("Professional Background Removal in 10 Lines of Code!", fontsize=14)
plt.tight_layout()
plt.show()

print("🤯 This is EXACTLY how Zoom/Teams/Google Meet works!")
print("   You just built a $10M feature in 10 lines!")

## 🎨 Interactive Background Color Selector

Play with different backgrounds. Notice how the mask stays perfect regardless of color!

In [ ]:
@widgets.interact(
    bg_color=widgets.Dropdown(
        options=[('White', (255,255,255)), ('Black', (0,0,0)), 
                ('Green Screen', (0,255,0)), ('Blue', (0,0,255))],
        value=(255,255,255),
        description='BG Color:'
    )
)
def interactive_bg_removal(bg_color):
    result = remove_background('street.jpg', seg_model, bg_color=bg_color)
    
    plt.figure(figsize=(10, 6))
    plt.imshow(result)
    plt.axis('off')
    plt.title(f'Background: RGB{bg_color}')
    plt.show()

## 🧠 What You Just Built

This isn't just a toy - this is the SAME technology used by:
- Zoom/Teams for virtual backgrounds
- Photo editors for cutouts
- E-commerce for product photography
- Film industry for green screen effects

**Now let's see it live on YOU!**

# 🎥 DEMO 2: Live Segmentation - "Remove My Background!"

In [ ]:
def live_segmentation_demo():
    """Live background removal with webcam"""
    try:
        model = YOLO('yolov8n-seg.pt')
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            print("📹 Webcam not available. Using simulation.")
            return simulate_segmentation_demo()
        
        print("\n" + "="*60)
        print("🎥 LIVE SEGMENTATION - BACKGROUND REMOVAL")
        print("="*60)
        print("\n• Watch your background disappear!")
        print("• This is how Zoom/Teams works")
        print("\n⚠️ Press 'q' to quit\n")
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Run segmentation
            results = model(frame, verbose=False)[0]
            
            if len(results.masks) > 0:
                # Create mask for first person (class 0)
                h, w = frame.shape[:2]
                combined_mask = np.zeros((h, w), dtype=bool)
                
                for mask, cls in zip(results.masks.data, results.boxes.cls):
                    if cls == 0:  # person
                        mask_np = mask.cpu().numpy()
                        mask_resized = cv2.resize(mask_np, (w, h))
                        combined_mask = np.logical_or(combined_mask, mask_resized > 0.5)
                
                # Apply mask
                result = frame.copy()
                result[~combined_mask] = [255, 255, 255]
            else:
                result = frame
            
            cv2.putText(result, "Background Removed - SAIR", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(result, "Press 'q' to quit", (10, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
            cv2.imshow('YOLO Segmentation - SAIR', result)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        print("\n✅ Demo complete!")
        
    except Exception as e:
        print(f"⚠️ Demo error: {e}")
        simulate_segmentation_demo()

def simulate_segmentation_demo():
    """Fallback simulation"""
    print("\n🎥 SIMULATED SEGMENTATION DEMO")
    model = YOLO('yolov8n-seg.pt')
    results = model('street.jpg')[0]
    
    plt.figure(figsize=(10,6))
    plt.imshow(results.plot())
    plt.title("Simulated Segmentation - Original Image with Masks")
    plt.axis('off')
    plt.show()

# RUN THIS FOR LIVE SEGMENTATION!
live_segmentation_demo()

## 🎯 From Pixels to People

We've gone from:
- **Boxes** (5A) → Rough location
- **Masks** (Now) → Exact shape

But what's NEXT? Understanding the **structure** of what we see.

**Think about it:**
- A mask tells us "this blob is a person"
- But WHERE are their arms? Head? Legs?

**Pose estimation answers: "Here are the joints!"**

# 🦾 PART 6.5: Pose Estimation - Can AI See How You Move?

## 🎯 From Pixels to People to Joints

We've come a long way:
- **5A:** Detecting objects (boxes)
- **Part 6:** Segmenting objects (pixel masks)
- **NOW:** Understanding human structure (joints)

**This is the bridge to gesture control!**

In [ ]:
# Load pose model
pose_model = YOLO('yolov8n-pose.pt')
print("✅ YOLOv8n-pose loaded")

# Run on image
pose_results = pose_model('street.jpg')[0]

# Visualize
plt.figure(figsize=(12, 8))
plt.imshow(pose_results.plot())
plt.axis('off')
plt.title(f'YOLO Pose: {len(pose_results.keypoints)} people with 17 keypoints each')
plt.show()

print(f"\n🎯 Detected {len(pose_results.keypoints)} people")
print("   Each person has 17 keypoints (joints)")

## 🔬 Understanding the 17 Keypoints

Each number corresponds to a specific body part:

```
0: Nose          9:  Left Wrist
1: Left Eye      10: Right Wrist
2: Right Eye     11: Left Hip
3: Left Ear      12: Right Hip
4: Right Ear     13: Left Knee
5: Left Shoulder 14: Right Knee
6: Right Shoulder15: Left Ankle
7: Left Elbow    16: Right Ankle
8: Right Elbow
```

**This is the vocabulary of human movement!**

## 🔗 Why Pose Matters

Pose estimation isn't just cool visualization - it's the foundation for:
1. **Gesture Control** (coming next!) - Understanding hand movements
2. **Action Recognition** - Detecting if someone is walking, running, sitting
3. **Fitness Tracking** - Counting reps, checking form
4. **Sports Analysis** - Analyzing athlete movement

The 17 keypoints you just saw are the vocabulary for understanding human movement!

## 🤔 Think About It

If you know WHERE all the joints are, what can you figure out?
- Arms up → Waving
- Knees bent → Sitting
- One knee down → Kneeling
- Arms wide → Open stance

**This is exactly how we'll build gesture control!**

# 🎥 DEMO 3: Live Pose - "YOLO Sees My Joints!"

In [ ]:
def live_pose_demo():
    """Live pose estimation with webcam"""
    try:
        model = YOLO('yolov8n-pose.pt')
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            print("📹 Using sample video instead...")
            return simulate_pose_demo()
        
        print("\n" + "="*60)
        print("🎥 LIVE POSE ESTIMATION - AI SEES YOUR JOINTS!")
        print("="*60)
        print("\n• Wave your arms")
        print("• Do a dance")
        print("• Strike a pose")
        print("\n⚠️ Press 'q' to quit\n")
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Run pose
            results = model(frame, verbose=False)[0]
            
            # Annotate
            annotated = results.plot()
            
            cv2.putText(annotated, f"People: {len(results.keypoints)}", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(annotated, "17 Keypoints Each", (10, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            cv2.putText(annotated, "Press 'q' to quit", (10, 90),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
            cv2.imshow('YOLO Pose - SAIR', annotated)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        print("\n✅ Demo complete!")
        
    except Exception as e:
        print(f"⚠️ Demo error: {e}")
        simulate_pose_demo()

def simulate_pose_demo():
    """Fallback simulation"""
    print("\n🎥 SIMULATED POSE DEMO")
    model = YOLO('yolov8n-pose.pt')
    results = model('street.jpg')[0]
    
    plt.figure(figsize=(10,6))
    plt.imshow(results.plot())
    plt.title("Simulated Pose - Sample Image")
    plt.axis('off')
    plt.show()

# RUN THIS FOR LIVE POSE!
live_pose_demo()

## 🧠 What You Just Experienced

The AI just tracked YOUR movements in real-time!

**This technology powers:**
- Fitness apps (count reps, check form)
- Dance games (Just Dance, etc.)
- Physical therapy (track recovery)
- Sports analysis (analyze athlete form)

**But we're not done yet...**

## 🎯 The Final Frontier

We can now:
1. ✅ Detect objects (boxes)
2. ✅ Segment objects (masks) 
3. ✅ Track joints (pose)

**What's missing? INTERACTION!**

Instead of just SEEING movement, let's UNDERSTAND what it means...

# ✨ PART 6.6: GRAND FINALE - Gesture Control

## 🎮 From Seeing to Understanding to INTERACTION

This is where everything comes together:
1. **CNN theory** (Lecture 3) → Feature extraction
2. **Transfer learning** (Lecture 4) → Efficient training
3. **Detection** (5A) → Finding people
4. **Pose estimation** (Part 6.5) → Finding joints
5. **Gesture control** (NOW) → Understanding intent

**YOU become the controller!**

In [ ]:
def gesture_control_demo():
    """Control your computer with hand gestures!"""
    try:
        model = YOLO('yolov8n-pose.pt')
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            print("📹 Using simulated gestures...")
            return simulate_gesture_demo()
        
        print("\n" + "="*60)
        print("🤚 GESTURE CONTROL - YOU ARE THE CONTROLLER!")
        print("="*60)
        print("\n🎮 GESTURES:")
        print("   • ✋ OPEN HAND: Stop/Pause")
        print("   • 👆 INDEX UP: Volume Up")
        print("   • ✌️ PEACE SIGN: Volume Down")
        print("   • 👊 FIST: Select/Click")
        print("   • 👍 THUMBS UP: Like!")
        print("\n⚠️ Press 'q' to quit\n")
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Run pose
            results = model(frame, verbose=False)[0]
            
            annotated = frame.copy()
            gesture = "No gesture detected"
            color = (255, 255, 255)
            
            if len(results.keypoints) > 0:
                # Get first person's keypoints
                kps = results.keypoints.data[0].cpu().numpy()
                
                # Extract wrist positions
                left_wrist = kps[9]   # Left wrist
                right_wrist = kps[10] # Right wrist
                
                if left_wrist[2] > 0.5 and right_wrist[2] > 0.5:
                    # Calculate distance between wrists
                    distance = np.sqrt(
                        (left_wrist[0] - right_wrist[0])**2 + 
                        (left_wrist[1] - right_wrist[1])**2
                    )
                    
                    # Simple gesture detection
                    if distance < 50:
                        gesture = "👊 FIST (Select/Click)"
                        color = (0, 0, 255)
                    elif distance > 200:
                        gesture = "✋ OPEN HAND (Stop/Pause)"
                        color = (0, 255, 0)
                    elif left_wrist[1] < right_wrist[1] - 30:
                        gesture = "👆 INDEX UP (Volume Up)"
                        color = (255, 0, 0)
                    elif right_wrist[1] < left_wrist[1] - 30:
                        gesture = "✌️ PEACE SIGN (Volume Down)"
                        color = (255, 255, 0)
                    else:
                        # Check thumb up (simplified)
                        if left_wrist[1] < kps[5][1] - 30:
                            gesture = "👍 THUMBS UP (Like!)"
                            color = (0, 255, 255)
                
                # Draw keypoints for visualization
                for i, (x, y, conf) in enumerate(kps):
                    if conf > 0.5:
                        cv2.circle(annotated, (int(x), int(y)), 5, (0, 255, 0), -1)
            
            # Display gesture
            cv2.putText(annotated, gesture, (50, 100),
                       cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 3)
            cv2.putText(annotated, "Press 'q' to quit", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
            cv2.imshow('Gesture Control - SAIR', annotated)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        print("\n✅ Demo ended. You just controlled a computer with your HANDS!")
        
    except Exception as e:
        print(f"⚠️ Demo error: {e}")
        simulate_gesture_demo()

def simulate_gesture_demo():
    """Fallback simulation"""
    print("\n🎥 SIMULATED GESTURE DEMO")
    print("\nDetected gestures would appear here:")
    gestures = ["✋ OPEN HAND", "👆 INDEX UP", "✌️ PEACE SIGN", "👊 FIST", "👍 THUMBS UP"]
    for g in gestures:
        print(f"   {g}")
        time.sleep(0.5)

# RUN THIS FOR THE GRAND FINALE!
gesture_control_demo()

## 🧠 Understanding the Gesture Pipeline

```
Webcam Frame → YOLO Pose → Keypoints → Distance Calculations → Gesture Classification
     ↓              ↓           ↓                ↓                      ↓
   Raw         17 joint    (x,y) for      Wrist distance,      "FIST", "OPEN",
  pixels      positions   each joint   relative positions     "THUMBS UP"
```

Each step builds on the previous. You've mastered them all!

## 🎉 YOU DID IT!

You just built a gesture-controlled system!

**This is the same technology used in:**
- Microsoft Kinect
- VR headsets (hand tracking)
- Smart TVs (gesture control)
- Sign language translation
- Touchless interfaces (hospitals, factories)

**But wait... we used PRE-TRAINED models.**

What if you want to detect YOUR OWN custom gestures? Like a secret hand sign?

## 🎓 The Million-Dollar Question

Everyone can use pre-trained models. 
**But YOU will learn to train custom models.**

This is what separates:
- Hobbyists (use existing models) from
- Professionals (create custom solutions)

**Let's learn the #1 skill in computer vision...**

# 🚀 PART 7: THE MAIN EVENT - Train YOLO on YOUR Data

## 🎓 This is the #1 Skill in Computer Vision

**Everyone can use pre-trained models.**  
**YOU will train custom models.**

In [ ]:
print("="*60)
print("PART 7: TRAIN YOLO ON YOUR DATA")
print("="*60)

# Download practice dataset
!wget -q -O coco128.zip https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip
!unzip -q -o coco128.zip -d datasets/
print("✅ Downloaded COCO128 practice dataset")

In [ ]:
import yaml

# Create dataset YAML
dataset_path = os.path.abspath('datasets/coco128')
data_yaml = {
    'path': dataset_path,
    'train': 'images/train2017',
    'val': 'images/train2017',
    'nc': 80,
    'names': ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat',
              'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat',
              'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack',
              'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
              'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
              'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
              'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair',
              'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse',
              'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
              'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
}

with open('coco128.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print("✅ Created coco128.yaml")

## 📊 What's in a YAML file?

This file tells YOLO:
- `path`: Where to find your data
- `train`: Which folder has training images
- `val`: Which folder has validation images  
- `nc`: Number of classes (80 for COCO)
- `names`: What to call each class

**For YOUR custom dataset, you'll change these to match YOUR objects!**

# 📸 PART 7.5: DATA COLLECTION MASTERCLASS

## Your Phone is Your Dataset Generator

**Best Practices:**
```
📱 iPhone/Android Photos → 1080p minimum
☀️ Lighting: Natural daylight best
🎭 Variety: 5+ different backgrounds
📐 Angles: Front, side, top-down
📏 Scale: Close-ups and far shots
🔄 Rotation: 0°, 90°, 180°, 270°
```

In [ ]:
def validate_yolo_annotations(image_dir, label_dir):
    """Check annotation format and quality"""
    issues = []
    
    for img_file in Path(image_dir).glob('*.jpg'):
        label_file = Path(label_dir) / f"{img_file.stem}.txt"
        
        # Check label exists
        if not label_file.exists():
            issues.append(f"⚠️ Missing label for {img_file.name}")
            continue
        
        # Check format and values
        with open(label_file) as f:
            lines = f.readlines()
            if not lines:
                issues.append(f"⚠️ Empty label file: {label_file.name}")
                continue
            
            for i, line in enumerate(lines):
                parts = line.strip().split()
                if len(parts) != 5:
                    issues.append(f"❌ Bad format in {label_file.name}: line {i+1}")
                    continue
                
                try:
                    # Check class ID is integer
                    class_id = int(parts[0])
                    # Check values in [0,1]
                    values = [float(x) for x in parts[1:]]
                    if not all(0 <= v <= 1 for v in values):
                        issues.append(f"❌ Values out of range in {label_file.name}: line {i+1}")
                except ValueError:
                    issues.append(f"❌ Non-numeric values in {label_file.name}: line {i+1}")
    
    # Report results
    if issues:
        print("\n🔍 Annotation Issues Found:")
        for issue in issues[:10]:  # Show first 10
            print(issue)
        if len(issues) > 10:
            print(f"... and {len(issues)-10} more issues")
    else:
        print("✅ All annotations look good!")
    
    return len(issues) == 0

# Test on coco128 dataset
validate_yolo_annotations('datasets/coco128/images/train2017', 
                         'datasets/coco128/labels/train2017')

## 🔍 Why Validate Annotations?

The #1 reason custom training fails: **Bad annotations**

Common issues this function catches:
- Missing labels (forgot to annotate)
- Wrong format (used COCO instead of YOLO)
- Values outside [0,1] (forgot to normalize)
- Extra spaces or tabs (corrupt files)

**Always validate before training!**

## 🚀 THE MAGIC COMMAND

**Train YOUR model** (uncomment to run)

In [ ]:
# UNCOMMENT TO TRAIN (takes ~10-30 minutes)
# model = YOLO('yolov8n.pt')
# results = model.train(
#     data='coco128.yaml',
#     epochs=50,  # Increase to 100-300 for real projects
#     imgsz=640,
#     batch=8,
#     device=0 if torch.cuda.is_available() else 'cpu',
#     augment=True,
#     patience=50
# )
# print("✅ Training complete! Check runs/detect/train/")

## 🎯 What Happens During Training?

```
Epoch 1/50: Loss=3.2 (learning rough shapes)
Epoch 10/50: Loss=1.8 (learning details)
Epoch 25/50: Loss=1.1 (fine-tuning)
Epoch 50/50: Loss=0.8 (converged)
```

The model gradually improves from "seeing blobs" to "recognizing YOUR objects"!

# 🎥 DEMO 5: Victory Lap - "YOUR Model Detects Me!"

In [ ]:
def victory_lap():
    """Test your trained model live"""
    # Find best model
    import glob
    model_paths = glob.glob('/home/silva/SILVA.AI/Projects/SAIR/runs/detect/train/weights/best.pt')
    
    if not model_paths:
        print("❌ No trained model found. Run training first.")
        print("   Uncomment the training cell above!")
        return
    
    your_model = YOLO(model_paths[-1])  # Latest model
    print(f"✅ Loaded YOUR model from {model_paths[-1]}")
    
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("📹 Testing on sample image instead...")
        results = your_model('street.jpg')[0]
        plt.figure(figsize=(10,6))
        plt.imshow(results.plot())
        plt.title("YOUR Custom Model - Trained by YOU!")
        plt.axis('off')
        plt.show()
        return
    
    print("\n🎥 YOUR model detecting YOU in real-time!")
    print("Press 'q' to quit\n")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        results = your_model(frame, verbose=False)[0]
        annotated = results.plot()
        
        cv2.putText(annotated, "YOUR Trained Model!", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow('YOUR Model - SAIR', annotated)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print("\n🎉 YOU DID IT! You trained and deployed a custom model!")

# Run after training
victory_lap()

## 🎯 From Learning to Building

You now have ALL the skills to build real-world CV applications:

1. ✅ **Detection** (5A): Find objects
2. ✅ **Segmentation** (5B): Isolate objects  
3. ✅ **Pose** (5B): Track movement
4. ✅ **Gesture** (5B): Understand intent
5. ✅ **Custom Training** (5B): YOUR data, YOUR model

**Now it's time to build something YOURS!**

# 🎯 PART 10: Portfolio Projects & Final Challenge

## 🚀 8 Portfolio Projects - Pick One!

In [ ]:
projects = """
1. 🚗 Parking Lot Monitor
   - Detect available parking spots
   - Count cars in real-time
   - Alert when lot is full

2. 🛒 Shelf Scanner
   - Track inventory on shelves
   - Detect out-of-stock items
   - Alert when restocking needed

3. 🏭 Safety Gear Detector
   - Detect helmets, vests, glasses
   - Alert when worker missing PPE
   - Log safety compliance

4. 📦 Package Sorter
   - Sort packages by size/type
   - Detect damaged packages
   - Route to correct destination

5. 🐕 Pet Monitor
   - Detect when pet enters frame
   - Track pet activity
   - Send notifications

6. 🌿 Plant Disease Detector
   - Detect diseased leaves
   - Classify disease type
   - Recommend treatment

7. 🚲 Bike Lane Violation
   - Detect vehicles in bike lanes
   - Capture license plates
   - Generate reports

8. 🏋️ Exercise Form Checker
   - Check pose during exercise
   - Count reps automatically
   - Provide form feedback
"""

print(projects)
print("\n💡 Each project uses what you learned:")
print("   1. Collect 50-200 images (your phone!)")
print("   2. Annotate with makesense.ai (free)")
print("   3. Train YOLO using Part 7 code")
print("   4. Test and iterate")
print("   5. Deploy with Gradio (5 lines)")

## 🏆 THE FINAL CHALLENGE

In [ ]:
challenge = """
🎯 YOUR MISSION (This Week)
============================

1. Pick a project from above (or invent your own)
2. Take 50 photos with your phone
3. Annotate at makesense.ai (15 min)
4. Train YOLO using the code from Part 7
5. Test on new images
6. Create a Gradio demo:

```python
import gradio as gr
from ultralytics import YOLO

model = YOLO('path/to/your/best.pt')

def detect(image):
    results = model(image)
    return results[0].plot()

gr.Interface(
    fn=detect,
    inputs="image",
    outputs="image",
    title="MY Custom Detector - SAIR"
).launch()
```

7. Share in our channel with #MyFirstDetector

⏱️ Time: 2-3 hours
🏆 Winner: Most creative project gets featured!
"""

print(challenge)

# ✅ MASTERY CHECKLIST

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════╗
║                      MASTERY CHECKLIST                            ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║ Before moving to Lecture 5C, ensure you can:                     ║
║                                                                   ║
║ □ Segment objects with pixel-perfect masks (Part 6)              ║
║ □ Remove backgrounds like Zoom/Teams (Part 6.3)                  ║
║ □ Track 17 body keypoints (Part 6.5)                             ║
║ □ Build gesture recognition systems (Part 6.6)                   ║
║ □ Collect high-quality training data (Part 7.5)                  ║
║ □ Train YOLO on custom data (Part 7)                             ║
║ □ Validate annotations programmatically                          ║
║ □ Build a complete CV application (Part 10)                      ║
║                                                                   ║
║ 🎉 If you checked all: YOU'RE READY FOR LECTURE 5C!              ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
""")

# 📝 POST-LECTURE ASSESSMENT

In [ ]:
def post_lecture_quiz():
    """Final assessment"""
    questions = [
        {
            "q": "What's the main difference between detection and segmentation?",
            "options": [
                "Detection is faster",
                "Segmentation provides pixel-perfect masks",
                "Detection is more accurate",
                "Segmentation works on video"
            ],
            "correct": 1,
            "explanation": "Segmentation identifies exact pixels belonging to each object"
        },
        {
            "q": "How many keypoints does COCO pose estimation output?",
            "options": ["10", "17", "25", "50"],
            "correct": 1,
            "explanation": "COCO pose uses 17 keypoints for body joints"
        },
        {
            "q": "What's the #1 skill in computer vision according to this lecture?",
            "options": [
                "Using pre-trained models",
                "Training on custom data",
                "Tuning hyperparameters",
                "Deploying to production"
            ],
            "correct": 1,
            "explanation": "Training custom models on YOUR data is what separates pros from beginners"
        }
    ]
    
    score = 0
    print("\n" + "="*60)
    print("📝 POST-LECTURE MASTERY ASSESSMENT")
    print("="*60)
    
    for i, q in enumerate(questions, 1):
        print(f"\n{i}. {q['q']}")
        for j, opt in enumerate(q['options']):
            print(f"   {j}. {opt}")
        try:
            ans = int(input("Your answer: "))
            if ans == q['correct']:
                print("✅ Correct!")
                score += 1
            else:
                print(f"❌ Incorrect. {q['explanation']}")
        except:
            print("❌ Please enter a number")
    
    percentage = (score / len(questions)) * 100
    print(f"\n{'='*60}")
    print(f"FINAL SCORE: {score}/{len(questions)} ({percentage:.0f}%)")
    
    if percentage >= 80:
        print("🌟 EXCELLENT! You're ready for Lecture 5C!")
    elif percentage >= 60:
        print("👍 Good. Review missed concepts before 5C.")
    else:
        print("📚 Review this notebook before moving on.")

# Uncomment to take assessment
# post_lecture_quiz()

# 🎯 WHAT'S NEXT?

## Lecture 5C: Transformers for Vision 

You've mastered:
- ✅ Detection (5A)
- ✅ Segmentation, Pose, Gesture (5B)

Next, you'll learn:
- 🤖 **Vision Transformers** - ViT, DETR, Swin

**See you in 5C!** 🚀

---

<div style="text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 10px;">
    <h1>🌟 From Understanding to Interaction 🌟</h1>
    <p style="font-size: 18px; font-style: italic;">"You started with boxes. Now you control with gestures."</p>
    <p><strong>SAIR Community - Sudanese Artificial Intelligence Research</strong></p>
    <p>🇸🇩 Building Sudan's AI Future, One Model at a Time</p>
</div>